# F12-UC3 — YOLOv8 Urban Tree Detection: RGB vs RGB+NIR

**Project:** Micro-Forest Health Monitoring — NEUSTA France  
**Author:** Sofya Tadevosyan  
**Date:** April 2026  

This notebook trains and evaluates two YOLOv8 models:
- **Baseline:** YOLOv8s on standard RGB (3-channel) aerial images
- **Improvement:** YOLOv8s patched to accept RGB+NIR (4-channel) — exploiting the Near-Infrared band for vegetation health

**Runtime required:** GPU (T4 or better). Go to `Runtime → Change runtime type → T4 GPU`.

**Dataset:** NAIP urban tree detection dataset (Ventura et al., 2024) — 1,651 images, 96,547 annotated trees.

---
## Setup checklist before running
1. Runtime → Change runtime type → **T4 GPU**
2. Upload the dataset folder to Google Drive at `My Drive/NEUSTA/Dataset/urban-tree-detection-data-main/`
3. Runtime → Run all

## Cell 1 — Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU')

## Cell 2 — Install dependencies

In [ ]:
!pip install -q ultralytics rasterio opencv-python-headless tqdm PyYAML pandas matplotlib

## Cell 3 — Mount Google Drive and set paths

**Before running:** upload the dataset folder to Google Drive at:  
`My Drive/NEUSTA/Dataset/urban-tree-detection-data-main/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DATASET_DIR = '/content/drive/MyDrive/NEUSTA/Dataset/urban-tree-detection-data-main'
WORK_DIR    = '/content/micro_forest'
YOLO_DIR    = '/content/micro_forest/yolo_dataset'
RESULTS_DIR = '/content/micro_forest/results'

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

assert os.path.isdir(DATASET_DIR), f'Dataset not found at {DATASET_DIR}'
n_tifs = len([f for f in os.listdir(os.path.join(DATASET_DIR, 'images')) if f.endswith('.tif')])
print(f'Dataset found. TIF images: {n_tifs} (expected 1651)')

## Cell 4 — Clone the GitHub repo

In [ ]:
import os
import subprocess

REPO_DIR = '/content/repo'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone',
                    'https://github.com/SofiaTadevosyan/Micro-Forest-Neusta-Masters-Internship.git',
                    REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

SCRIPTS_DIR = os.path.join(REPO_DIR, 'yolov8_urban_trees')
print('Scripts available:', os.listdir(SCRIPTS_DIR))

## Cell 5 — Convert dataset to YOLO format

Reads all 1,651 TIFF files, exports RGB PNGs + RGBN .npy files, converts point annotations to YOLO boxes (radius=15 px).  
Then patches the YAML files to use absolute paths (required for Ultralytics to find labels correctly).  
**Takes ~5-10 minutes. Skipped automatically if already done.**

In [ ]:
import os
import subprocess

RGB_YAML  = os.path.join(YOLO_DIR, 'dataset_rgb.yaml')
RGBN_YAML = os.path.join(YOLO_DIR, 'dataset_rgbn.yaml')

if not os.path.exists(RGB_YAML):
    print('Converting dataset...')
    result = subprocess.run(
        ['python3', os.path.join(SCRIPTS_DIR, 'convert_annotations.py'),
         '--dataset_dir', DATASET_DIR,
         '--output_dir',  YOLO_DIR,
         '--radius', '15'],
        check=True
    )
    print('Conversion done.')
else:
    print('Dataset already converted. Skipping.')

# IMPORTANT: patch YAML files to use absolute paths.
# convert_annotations.py writes 'path: /content/micro_forest/yolo_dataset' already
# but we re-write both YAMLs here to be safe — relative paths cause box_loss=0.
import yaml

for yaml_path in [RGB_YAML, RGBN_YAML]:
    with open(yaml_path) as f:
        cfg = yaml.safe_load(f)
    cfg['path'] = YOLO_DIR
    with open(yaml_path, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)
    print(f'Patched path in {os.path.basename(yaml_path)} -> {YOLO_DIR}')

# Verify counts
for split in ['train', 'val', 'test']:
    rgb_dir  = os.path.join(YOLO_DIR, 'images', 'rgb',  split)
    rgbn_dir = os.path.join(YOLO_DIR, 'images', 'rgbn', split)
    lbl_dir  = os.path.join(YOLO_DIR, 'labels', split)
    n_rgb  = len(os.listdir(rgb_dir))  if os.path.isdir(rgb_dir)  else 0
    n_rgbn = len(os.listdir(rgbn_dir)) if os.path.isdir(rgbn_dir) else 0
    n_lbl  = len(os.listdir(lbl_dir))  if os.path.isdir(lbl_dir)  else 0
    print(f'{split:5s} -> RGB: {n_rgb:4d} PNGs | RGBN: {n_rgbn:4d} NPYs | Labels: {n_lbl:4d} TXTs')

## Cell 6 — Train RGB Baseline (YOLOv8s, 3-channel)

Standard YOLOv8s trained on RGB aerial images.  
**Expected time on T4 GPU: ~45-60 minutes for 50 epochs.**

In [ ]:
import os
import subprocess

RGB_WEIGHTS = '/content/micro_forest/runs/train/yolov8_rgb/weights/best.pt'

if not os.path.exists(RGB_WEIGHTS):
    print('Starting RGB training...')
    result = subprocess.run(
        ['python3', os.path.join(SCRIPTS_DIR, 'train_rgb.py'),
         '--data',    RGB_YAML,
         '--model',   'yolov8s.pt',
         '--epochs',  '50',
         '--imgsz',   '256',
         '--batch',   '32',
         '--name',    'yolov8_rgb',
         '--project', '/content/micro_forest/runs/train',
         '--device',  '0'],
        check=True
    )
    print('RGB training done. Return code:', result.returncode)
else:
    print('RGB model already trained. Skipping.')

print('RGB weights exist:', os.path.exists(RGB_WEIGHTS))

## Cell 7 — Train RGB+NIR Model (YOLOv8s, 4-channel)

YOLOv8s with first Conv2d patched to accept 4 channels (RGB + Near-Infrared).  
NIR filter initialised as mean of pretrained RGB filters.  
**Expected time on T4 GPU: ~60-75 minutes for 50 epochs.**

In [ ]:
import os
import subprocess

RGBN_WEIGHTS = '/content/micro_forest/runs/train/yolov8_rgbn/weights/best.pt'

if not os.path.exists(RGBN_WEIGHTS):
    print('Starting RGBN training...')
    result = subprocess.run(
        ['python3', os.path.join(SCRIPTS_DIR, 'train_rgbn.py'),
         '--data',    RGBN_YAML,
         '--model',   'yolov8s.pt',
         '--epochs',  '50',
         '--batch',   '16',
         '--name',    'yolov8_rgbn',
         '--project', '/content/micro_forest/runs/train',
         '--device',  'cuda',
         '--workers', '2'],
        check=True
    )
    print('RGBN training done. Return code:', result.returncode)
else:
    print('RGBN model already trained. Skipping.')

print('RGBN weights exist:', os.path.exists(RGBN_WEIGHTS))

## Cell 8 — Evaluate and Compare Both Models

- RGB model evaluated with Ultralytics `.val()` (standard, uses PNG images)
- RGBN model evaluated by direct inference on 4-channel NPY files (avoids channel mismatch)
- Saves `metrics.json`, `results_comparison.csv`, `results_comparison.png`

In [ ]:
import os
import subprocess

result = subprocess.run(
    ['python3', os.path.join(SCRIPTS_DIR, 'evaluate.py'),
     '--rgb_weights',  RGB_WEIGHTS,
     '--rgbn_weights', RGBN_WEIGHTS,
     '--data_rgb',     RGB_YAML,
     '--data_rgbn',    RGBN_YAML,
     '--output_dir',   RESULTS_DIR],
    check=True
)
print('Evaluation done. Return code:', result.returncode)

## Cell 9 — Display results

In [ ]:
import json
import os
import pandas as pd
from IPython.display import Image, display

metrics_path = os.path.join(RESULTS_DIR, 'metrics.json')
assert os.path.exists(metrics_path), f'metrics.json not found at {metrics_path}'

with open(metrics_path) as f:
    metrics = json.load(f)

df = pd.DataFrame(metrics).T
df.index = ['RGB (baseline)', 'RGB+NIR']
print('\n=== RESULTS ===')
print(df.to_string(float_format=lambda x: f'{x:.4f}'))

chart_path = os.path.join(RESULTS_DIR, 'results_comparison.png')
if os.path.exists(chart_path):
    display(Image(chart_path))
else:
    print('Chart not found:', chart_path)

## Cell 10 — Save results and weights to Google Drive

In [ ]:
import shutil
import os

DRIVE_OUTPUT  = '/content/drive/MyDrive/NEUSTA/Results'
DRIVE_WEIGHTS = os.path.join(DRIVE_OUTPUT, 'weights')
os.makedirs(DRIVE_OUTPUT,  exist_ok=True)
os.makedirs(DRIVE_WEIGHTS, exist_ok=True)

# Copy results (CSV, PNG, JSON)
for fname in os.listdir(RESULTS_DIR):
    src = os.path.join(RESULTS_DIR, fname)
    if os.path.isfile(src):
        shutil.copy(src, os.path.join(DRIVE_OUTPUT, fname))
        print(f'Copied: {fname}')

# Copy weights
if os.path.exists(RGB_WEIGHTS):
    shutil.copy(RGB_WEIGHTS, os.path.join(DRIVE_WEIGHTS, 'best_rgb.pt'))
    print('Copied: best_rgb.pt')
else:
    print('WARNING: RGB weights not found, skipping')

if os.path.exists(RGBN_WEIGHTS):
    shutil.copy(RGBN_WEIGHTS, os.path.join(DRIVE_WEIGHTS, 'best_rgbn.pt'))
    print('Copied: best_rgbn.pt')
else:
    print('WARNING: RGBN weights not found, skipping')

print(f'\nAll saved to: {DRIVE_OUTPUT}')

## Cell 11 — Show sample prediction visualisations

Draws ground truth boxes (green) and predicted boxes (red) on 4 sample test images.

In [ ]:
import os
import subprocess
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import Image, display

VIZ_DIR = os.path.join(RESULTS_DIR, 'viz')

# Generate visualisations if not already done
if not os.path.exists(VIZ_DIR) or len(os.listdir(VIZ_DIR)) == 0:
    print('Generating visualisations...')
    subprocess.run(
        ['python3', os.path.join(SCRIPTS_DIR, 'evaluate.py'),
         '--rgb_weights',  RGB_WEIGHTS,
         '--rgbn_weights', RGBN_WEIGHTS,
         '--data_rgb',     RGB_YAML,
         '--data_rgbn',    RGBN_YAML,
         '--output_dir',   RESULTS_DIR,
         '--visualise',
         '--n_samples', '4'],
        check=True
    )

if os.path.exists(VIZ_DIR):
    viz_files = sorted([f for f in os.listdir(VIZ_DIR) if f.startswith('rgb_')])[:4]
    if viz_files:
        fig, axes = plt.subplots(1, len(viz_files), figsize=(16, 4))
        if len(viz_files) == 1:
            axes = [axes]
        fig.suptitle('RGB Predictions — green=ground truth, red=predicted', fontsize=12)
        for ax, fname in zip(axes, viz_files):
            ax.imshow(mpimg.imread(os.path.join(VIZ_DIR, fname)))
            ax.set_title(fname.replace('rgb_', ''), fontsize=7)
            ax.axis('off')
        plt.tight_layout()
        DRIVE_OUTPUT = '/content/drive/MyDrive/NEUSTA/Results'
        save_path = os.path.join(DRIVE_OUTPUT, 'sample_predictions_rgb.png')
        plt.savefig(save_path, dpi=150)
        plt.show()
        print(f'Saved to Drive: {save_path}')
    else:
        print('No RGB visualisation files found in', VIZ_DIR)
else:
    print('Visualisation directory not found. Skipping.')